In [ ]:
import json
import os
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

True

# Metadata and helper functions

In [210]:
METADATA_CURRENT_TIME = time.time()

In [211]:
def xml_to_dataframe(xml_string):
    root = ET.fromstring(xml_string)
    data_type = next(
        elem for elem in root.iter()
        if elem.tag.split("}")[-1] == "dataType"
    )

    columns = [
        elem.text.strip()
        for elem in data_type
        if elem.tag.split("}")[-1] == "columns"
    ]

    records = []
    timestamp = None

    for record in data_type:
        if record.tag.split("}")[-1] != "record":
            continue

        row = []

        for child in record:
            tag = child.tag.split("}")[-1]

            if tag == "date":
                if timestamp is None:
                    timestamp = pd.to_datetime(child.text.strip(), utc=True)

            elif tag == "stringValue":
                row.append(child.text.strip())

        records.append(row)

    columns_without_time = [
        col for col in columns if col != "utc"
    ]


    df = pd.DataFrame(
        records,
        columns=columns_without_time
    )

    df.columns = df.columns.str.replace(' (deg.)', '')

    for column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    return df, timestamp

def print_section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def print_mapping(title, data):
    print(f"\n{title}:")
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {data}")


def print_latest(title, data):
    print(f"\n{title}:")

    if isinstance(data, list):
        if len(data) > 1:
            print(f"  Latest: {data[-1]}")
        else:
            print(f"  {data}")
    elif isinstance(data, dict):
        for key, value in data.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {data}")


def print_local_values(title, dataframe):
    local_values = dataframe[
        dataframe["longitude"].between(37, 38)
        & dataframe["latitude"].between(54, 56)
    ]

    print_section(title)

    if local_values.empty:
        print("No values available near Moscow.")
    else:
        print(local_values.to_string(index=False))


def dataframe_to_records(dataframe):
    return dataframe.where(dataframe.notna(), None).to_dict(orient="records")


def serialize_timestamp(value):
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return value

# Weather

In [212]:
moscow_lat_lon = (55.5269, 37.0888)
apikey = os.getenv("OPENWEATHER_API_KEY")
requeststring = f"https://api.openweathermap.org/data/2.5/weather?lat={moscow_lat_lon[0]}&lon={moscow_lat_lon[1]}&appid={apikey}"
weather_data = requests.get(requeststring)

In [213]:
pprint(weather_data.json())

{'base': 'stations',
 'clouds': {'all': 100},
 'cod': 200,
 'coord': {'lat': 55.5269, 'lon': 37.0888},
 'dt': 1788820742,
 'id': 6417504,
 'main': {'feels_like': 280.1,
          'grnd_level': 994,
          'humidity': 87,
          'pressure': 1017,
          'sea_level': 1017,
          'temp': 282.29,
          'temp_max': 282.29,
          'temp_min': 282.07},
 'name': 'Kromino',
 'sys': {'country': 'RU',
         'id': 9033,
         'sunrise': 1788835791,
         'sunset': 1788883764,
         'type': 1},
 'timezone': 10800,
 'visibility': 10000,
 'weather': [{'description': 'overcast clouds',
              'icon': '04n',
              'id': 804,
              'main': 'Clouds'}],
 'wind': {'deg': 290, 'speed': 4}}


# Ionosphere data
## vertical total electron content (VTEC)

grid is the gridded VTEC field that becomes the colored map

ipps are the GNSS ionospheric piercing points showing where the GNSS signal paths intersect the assumed ionospheric shell

In [214]:
# DLR_GNSS_GCG_L4_VTEC-NTCM-SCM_NC_EUROPE_latest_D.json

VTEC_EUROPE_DATA = requests.get("https://data.impc.dlr.de/tec-nowcast/DLR_GNSS_GCG_L4_VTEC-NTCM-SCM_NC_EUROPE/latest/DLR_GNSS_GCG_L4_VTEC-NTCM-SCM_NC_EUROPE_latest_D.json").json()
VTEC_GLOBAL_DATA = requests.get("https://data.impc.dlr.de/tec-nowcast/DLR_GNSS_GCG_L4_VTEC-NTCM-SCM_NC_GLOBAL/latest/DLR_GNSS_GCG_L4_VTEC-NTCM-SCM_NC_GLOBAL_latest_D.json").json()

In [215]:
import pandas as pd

features = VTEC_GLOBAL_DATA["data"]["grid"]["features"]

rows = []

for f in features:
    lon, lat = f["geometry"]["coordinates"]

    p = f["properties"]

    rows.append({
        "longitude": lon,
        "latitude": lat,
        "vtec_assimilated": p["vtec_assimilated_tecu"],
        "vtec_model": p["vtec_model_tecu"],
        "vtec_rms": p["vtec_rms_tecu"],
    })

grid = pd.DataFrame(rows)

print(grid.head())
print(grid.shape)

print("At Moscow (55.5269, 37.0888):")
print(grid[(grid["longitude"] >= 35) & (grid["longitude"] <= 40) & (grid["latitude"] >= 53) & (grid["latitude"] <= 57)])

   longitude  latitude  vtec_assimilated  vtec_model  vtec_rms
0     -180.0     -90.0               0.5       0.635      3.65
1     -175.0     -90.0               0.5       0.635      3.65
2     -170.0     -90.0               0.5       0.635      3.65
3     -165.0     -90.0               0.5       0.635      3.65
4     -160.0     -90.0               0.5       0.635      3.65
(5329, 5)
At Moscow (55.5269, 37.0888):
      longitude  latitude  vtec_assimilated  vtec_model  vtec_rms
4277       35.0      55.0             4.925       4.139       0.5
4278       40.0      55.0             4.910       4.294       0.5


In [216]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 8))

# sc = plt.scatter(
#     grid["longitude"],
#     grid["latitude"],
#     c=grid["vtec_assimilated"],
#     s=10,
#     cmap="turbo"
# )

# plt.colorbar(sc, label="VTEC [TECU]")

# plt.xlabel("Longitude (deg)")
# plt.ylabel("Latitude (deg)")
# plt.title("IMPC Europe VTEC")

# plt.show()

## SWE ESA
Europe is sufficient here

In [217]:
SWE_S4_DATA=requests.get("https://swe.ssa.esa.int/assets/ism/ism_nowcast/s4_nowcast_grid.xml").text
SWE_SIGMA_PHI_DATA=requests.get("https://swe.ssa.esa.int/assets/ism/ism_nowcast/sigma_phi_nowcast_grid.xml").text
SWE_TEC_DATA = requests.get("https://swe.ssa.esa.int/assets/ism/ism_nowcast/tec_nowcast_grid.xml").text

In [218]:
df_swe_s4, ts = xml_to_dataframe(SWE_S4_DATA)
df_swe_sigma_phi, ts = xml_to_dataframe(SWE_SIGMA_PHI_DATA)
df_swe_tec, ts = xml_to_dataframe(SWE_TEC_DATA)

for df in [df_swe_s4, df_swe_sigma_phi, df_swe_tec]:
    print(df.head())
    print("At Moscow (55.5269, 37.0888):")
    print(df[(df["longitude"] >= 37) & (df["longitude"] <= 38) &
                (df["latitude"] >= 54) & (df["latitude"] <= 56)])
    print()
    print()
    print()


   longitude  latitude     s4  s4_error
0     -180.0     -87.5  0.012     0.046
1     -177.5     -87.5  0.013     0.046
2     -175.0     -87.5  0.013     0.046
3     -172.5     -87.5  0.013     0.046
4     -170.0     -87.5  0.014     0.046
At Moscow (55.5269, 37.0888):
      longitude  latitude     s4  s4_error
8352       37.5      55.0  0.045     0.033



   longitude  latitude  sigma_phi (rad.)  sigma_phi_error (rad.)
0     -180.0     -87.5             0.004                   0.035
1     -177.5     -87.5             0.004                   0.035
2     -175.0     -87.5             0.005                   0.035
3     -172.5     -87.5             0.005                   0.035
4     -170.0     -87.5             0.005                   0.035
At Moscow (55.5269, 37.0888):
      longitude  latitude  sigma_phi (rad.)  sigma_phi_error (rad.)
8352       37.5      55.0             0.034                   0.026



   longitude  latitude  tec (TECU)  tec_error (TECU)
0     -180.0     -87.5       

## NOAA Solar activity

In [219]:
# Solar wind speed
NOAA_SOLAR_WIND_SPEED_DATA = requests.get("https://services.swpc.noaa.gov/products/summary/solar-wind-speed.json").json()

print(NOAA_SOLAR_WIND_SPEED_DATA) # km/sec

[{'proton_speed': 479, 'time_tag': '2026-09-07T22:32:00Z'}]


In [220]:
# 10cm flux
NOAA_10CM_FLUX_DATA = requests.get("https://services.swpc.noaa.gov/products/summary/10cm-flux.json").json()

print(NOAA_10CM_FLUX_DATA) # sfu

[{'flux': 110, 'time_tag': '2026-09-07T20:00:00'}]


In [221]:
NOAA_SOLAR_WIND_MAGNETIC_FIELD_DATA = requests.get("https://services.swpc.noaa.gov/products/summary/solar-wind-mag-field.json").json()

print(NOAA_SOLAR_WIND_MAGNETIC_FIELD_DATA) # nT
# bz is the north/south component of the interplanetary magnetic field in GSM coordinates.

[{'bt': 12, 'bz_gsm': -5, 'time_tag': '2026-09-07T22:32:00Z'}]


In [222]:
NOAA_SCALES_DATA = requests.get("https://services.swpc.noaa.gov/products/noaa-scales.json").json()

pprint(NOAA_SCALES_DATA['0']) # current

{'DateStamp': '2026-09-07',
 'G': {'Scale': '0', 'Text': 'none'},
 'R': {'MajorProb': None, 'MinorProb': None, 'Scale': '0', 'Text': 'none'},
 'S': {'Prob': None, 'Scale': '0', 'Text': 'none'},
 'TimeStamp': '22:37:00'}


In [223]:
# I already have

# Regular weather:
# - Current weather for Moscow coordinates (55.5269, 37.0888) from OpenWeather.
# - Raw JSON response includes temperature, pressure, humidity, wind, clouds,
#   visibility, weather conditions, sunrise/sunset, and related fields.
# - The API key is loaded from the OPENWEATHER_API_KEY environment variable.
#
# Space weather:
# - DLR global and European VTEC ionosphere grids.
# - Grid latitude/longitude points with assimilated VTEC, model VTEC,
#   and VTEC RMS values, measured in TECU.
# - A plotted global VTEC map.
#
# - ESA SWE Europe nowcast grids:
#   - S4 scintillation index.
#   - Sigma-phi phase scintillation index.
#   - TEC.
# - ESA XML data is parsed into pandas DataFrames with latitude, longitude,
#   numeric measurements, and a UTC timestamp.
# - Values near Moscow are queried from the ESA grids.
#
# - NOAA solar-wind speed summary, measured in km/s.
# - NOAA 10 cm solar flux, measured in SFU.
# - NOAA solar-wind magnetic-field data, measured in nT, including Bz.
# - NOAA current space-weather scales.
#
# GFZ Kp index
# GFZ HP30 index
#
# Kyoto Dst index


# TODO:
# Ap
# Dst
# AE
# local magnetic field
# local dB/dt

## Hemholtz center (GFZ)

In [224]:
GFZ_KP_DATA = requests.get("https://kp.gfz.de/app/json/kpnowcast.json").json()
pprint(GFZ_KP_DATA) # unix timestamp, kp

[[1788744600000, 1.0],
 [1788755400000, 1.667],
 [1788766200000, 3.0],
 [1788777000000, 3.333],
 [1788787800000, 3.0],
 [1788798600000, 3.667],
 [1788809400000, 3.667],
 [1788820200000, 3.667]]


In [225]:
GFZ_HP30_DATA = requests.get("https://kp.gfz.de/app/json/hpo30nowcast.json").json()

pprint(GFZ_HP30_DATA) # unix timestamp, hp30

[[1788736500000, 1.0],
 [1788738300000, 1.0],
 [1788740100000, 1.667],
 [1788741900000, 1.667],
 [1788743700000, 1.0],
 [1788745500000, 1.0],
 [1788747300000, 0.034],
 [1788749100000, 1.0],
 [1788750900000, 1.667],
 [1788752700000, 2.333],
 [1788754500000, 2.333],
 [1788756300000, 2.0],
 [1788758100000, 1.333],
 [1788759900000, 2.0],
 [1788761700000, 1.667],
 [1788763500000, 2.0],
 [1788765300000, 2.333],
 [1788767100000, 2.667],
 [1788768900000, 3.0],
 [1788770700000, 3.667],
 [1788772500000, 3.0],
 [1788774300000, 3.333],
 [1788776100000, 3.667],
 [1788777900000, 3.667],
 [1788779700000, 3.667],
 [1788781500000, 4.0],
 [1788783300000, 3.333],
 [1788785100000, 2.667],
 [1788786900000, 2.333],
 [1788788700000, 3.333],
 [1788790500000, 3.667],
 [1788792300000, 3.333],
 [1788794100000, 3.667],
 [1788795900000, 4.667],
 [1788797700000, 3.667],
 [1788799500000, 5.0],
 [1788801300000, 4.333],
 [1788803100000, 3.667],
 [1788804900000, 4.333],
 [1788806700000, 4.333],
 [1788808500000, 3.667],

## Kyoto

In [226]:
KYOTO_DST_DATA = requests.get("https://services.swpc.noaa.gov/products/kyoto-dst.json").json()
print(KYOTO_DST_DATA[-1])

{'time_tag': '2026-09-07T22:00:00', 'dst': -42}


In [227]:
# =====================================================================
# REGULAR WEATHER
# =====================================================================

print_section("REGULAR WEATHER")

weather = weather_data.json()
weather_main = weather.get("main", {})
weather_wind = weather.get("wind", {})
weather_coordinates = weather.get("coord", {})
weather_system = weather.get("sys", {})

temperature_c = weather_main.get("temp")
feels_like_c = weather_main.get("feels_like")

if temperature_c is not None:
    temperature_c = temperature_c - 273.15

if feels_like_c is not None:
    feels_like_c = feels_like_c - 273.15

print(f"Location: {weather.get('name', 'Unknown')}")
print(f"Coordinates: {weather_coordinates}")
print(f"Conditions: {weather.get('weather')}")
print(f"Temperature: {temperature_c:.1f} °C")
print(f"Feels like: {feels_like_c:.1f} °C")
print(f"Pressure: {weather_main.get('pressure')} hPa")
print(f"Humidity: {weather_main.get('humidity')}%")
print(f"Wind speed: {weather_wind.get('speed')} m/s")
print(f"Wind direction: {weather_wind.get('deg')}°")
print(f"Cloudiness: {weather.get('clouds', {}).get('all')}%")
print(f"Visibility: {weather.get('visibility')} m")
print(f"Sunrise: {weather_system.get('sunrise')}")
print(f"Sunset: {weather_system.get('sunset')}")


# =====================================================================
# IONOSPHERIC CONDITIONS
# =====================================================================

print_section("IONOSPHERIC CONDITIONS")

print("DLR VTEC:")
print(f"  Global grid points: {len(grid):,}")

print("\n  Global VTEC statistics:")
print(
    grid[
        ["vtec_assimilated", "vtec_model", "vtec_rms"]
    ].describe().to_string()
)

moscow_vtec = grid[
    grid["longitude"].between(35, 40)
    & grid["latitude"].between(53, 57)
]

print("\n  VTEC values near Moscow:")
if moscow_vtec.empty:
    print("    No values available.")
else:
    print(moscow_vtec.to_string(index=False))

print_local_values("ESA S4 SCINTILLATION NEAR MOSCOW", df_swe_s4)
print_local_values(
    "ESA SIGMA-PHI PHASE SCINTILLATION NEAR MOSCOW",
    df_swe_sigma_phi,
)
print_local_values("ESA TEC NEAR MOSCOW", df_swe_tec)

print(f"\nESA data timestamp: {ts}")


# =====================================================================
# GEOMAGNETIC CONDITIONS
# =====================================================================

print_section("GEOMAGNETIC CONDITIONS")

print_latest("GFZ Kp index", GFZ_KP_DATA)
print_latest("GFZ Hp30 index", GFZ_HP30_DATA)
print_latest("Kyoto Dst index", KYOTO_DST_DATA)


# =====================================================================
# SOLAR WIND AND INTERPLANETARY MAGNETIC FIELD
# =====================================================================

print_section("SOLAR WIND AND MAGNETIC FIELD")

print_mapping(
    "Solar-wind speed",
    NOAA_SOLAR_WIND_SPEED_DATA,
)

print_mapping(
    "Solar-wind magnetic field",
    NOAA_SOLAR_WIND_MAGNETIC_FIELD_DATA,
)


# =====================================================================
# SOLAR ACTIVITY AND SPACE-WEATHER SCALES
# =====================================================================

print_section("SOLAR ACTIVITY AND SPACE-WEATHER SCALES")

print_mapping(
    "10 cm solar flux",
    NOAA_10CM_FLUX_DATA,
)

print_mapping(
    "NOAA space-weather scales",
    NOAA_SCALES_DATA,
)


# =====================================================================
# DATA SOURCES SUMMARY
# =====================================================================

print_section("DATA SOURCES")

print("Regular weather:")
print("  - OpenWeather current conditions for Moscow")

print("\nIonospheric conditions:")
print("  - DLR global VTEC")
print("  - ESA S4 scintillation")
print("  - ESA sigma-phi phase scintillation")
print("  - ESA TEC")

print("\nGeomagnetic conditions:")
print("  - GFZ Kp")
print("  - GFZ Hp30")
print("  - Kyoto Dst")

print("\nSolar wind and magnetic field:")
print("  - NOAA solar-wind speed")
print("  - NOAA solar-wind magnetic field")

print("\nSolar activity:")
print("  - NOAA 10 cm solar flux")
print("  - NOAA space-weather scales")

print("\nMiscellaneous:")
print("  - Current timestamp:", METADATA_CURRENT_TIME)
print(f"Time it took to run the script: {time.time() - METADATA_CURRENT_TIME:2f} seconds")


REGULAR WEATHER
Location: Kromino
Coordinates: {'lon': 37.0888, 'lat': 55.5269}
Conditions: [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04n'}]
Temperature: 9.1 °C
Feels like: 7.0 °C
Pressure: 1017 hPa
Humidity: 87%
Wind speed: 4 m/s
Wind direction: 290°
Cloudiness: 100%
Visibility: 10000 m
Sunrise: 1788835791
Sunset: 1788883764

IONOSPHERIC CONDITIONS
DLR VTEC:
  Global grid points: 5,329

  Global VTEC statistics:
       vtec_assimilated   vtec_model     vtec_rms
count       5329.000000  5329.000000  5329.000000
mean          11.418425    12.428504     1.104058
std           11.379099    11.370505     1.111208
min            0.500000     0.500000     0.500000
25%            2.198000     3.383000     0.500000
50%            7.894000     9.349000     0.592000
75%           16.858000    17.827000     0.837000
max           49.396000    53.198000     4.167000

  VTEC values near Moscow:
 longitude  latitude  vtec_assimilated  vtec_model  vtec_rms
      35.0 

# Conclusions, packaging

In [228]:
# Alert if thundering

if any(w.get("main") == "Thunderstorm" for w in weather_data.json()["weather"]):
    print("CRITICAL: THUNDER")
    exit()


In [229]:
current_unix_timestamp = int(time.time())
output_path = f"{current_unix_timestamp}.json"


all_data = {
    "metadata": {
        "created_at_unix": current_unix_timestamp,
        "created_at_utc": serialize_timestamp(
            pd.Timestamp.now(tz="UTC")
        ),
        "location": {
            "name": "Moscow",
            "latitude": moscow_lat_lon[0],
            "longitude": moscow_lat_lon[1],
        },
    },

    "regular_weather": weather_data.json(),

    "ionospheric_conditions": {
        "dlr_vtec_europe": VTEC_EUROPE_DATA,
        "dlr_vtec_global": VTEC_GLOBAL_DATA,
        "dlr_vtec_global_grid": dataframe_to_records(grid),

        "esa_s4_grid": dataframe_to_records(df_swe_s4),
        "esa_sigma_phi_grid": dataframe_to_records(df_swe_sigma_phi),
        "esa_tec_grid": dataframe_to_records(df_swe_tec),
        "esa_data_timestamp": serialize_timestamp(ts),
    },

    "geomagnetic_conditions": {
        "gfz_kp": GFZ_KP_DATA,
        "gfz_hp30": GFZ_HP30_DATA,
        "kyoto_dst": KYOTO_DST_DATA,
    },

    "solar_wind_and_magnetic_field": {
        "solar_wind_speed": NOAA_SOLAR_WIND_SPEED_DATA,
        "solar_wind_magnetic_field": NOAA_SOLAR_WIND_MAGNETIC_FIELD_DATA,
    },

    "solar_activity": {
        "noaa_10cm_flux": NOAA_10CM_FLUX_DATA,
        "noaa_space_weather_scales": NOAA_SCALES_DATA,
    },

    "raw_esa_xml": {
        "s4": SWE_S4_DATA,
        "sigma_phi": SWE_SIGMA_PHI_DATA,
        "tec": SWE_TEC_DATA,
    },
}




with zipfile.ZipFile(f"{output_path}.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr(
        output_path,
        json.dumps(
            all_data,
            indent=2,
            ensure_ascii=False,
            default=serialize_timestamp,
        ),
    )




print("Saved collected data")

Saved collected data
